# FDM-PINN plate bending under centered rectangular patch load, FFFF

In [1]:
"""
FDM-PINN (Finite-Difference-Method PINN) for Kirchhoff-Love plate bending
under a PARTIALLY-DISTRIBUTED (centered rectangular patch) load, all-edges-
CLAMPED / built-in / fixed (BC case "FFFF").

This is derived from fdm_pinn_partial_patch_hardBC.py (the SS-SS-SS-SS
script) with the same two changes made for the sinusoidal/triangular FFFF
scripts (pinn_sinusoidal_FFFF_clamped.py, pinn_triangular_FFFF_clamped.py):

  1. The hard-BC envelope now enforces BOTH essential clamped conditions
     (w = 0 AND dw/dn = 0) on all four edges, instead of just w = 0.
  2. There is no closed-form "analytical" solution used here for the
     reference field (see note below) -- error metrics are instead
     computed against a numerical finite-difference solve of the actual
     governing biharmonic equation.

The FDM-PINN derivative machinery itself (ghost-extended grid, Fornberg
central-difference stencils, ONE forward pass per epoch, ordinary
.backward() instead of chained create_graph=True autograd) is UNCHANGED
from the vanilla FDM-PINN script -- see that script's docstring for what
FDM-PINN means and its caveats (O(h^4) truncation error, GHOST=3 minimum,
uniform-grid assumption, etc.). All of that still applies verbatim here.

-----------------------------------------------------------------------------
IMPORTANT: TWO DIFFERENT USES OF "FINITE DIFFERENCES" IN THIS FILE --
DO NOT CONFUSE THEM
-----------------------------------------------------------------------------
  (a) `fd_derivatives(model)` -- the FDM-PINN trick: differentiates the
      TRAINED NETWORK's single forward pass on a ghost-extended grid, used
      every epoch inside `compute_loss()`. This is what makes the script
      "FDM-PINN" as opposed to vanilla/autograd PINN.
  (b) `solve_fd_reference_clamped(...)` -- a completely separate, one-time
      finite-difference SOLVE of the governing PDE itself (not of a trained
      network), used only to generate the reference field for error
      metrics, replacing the closed-form analytical solution used in the
      SS-SS-SS-SS script. It has nothing to do with training.
Both happen to use finite-difference stencils, which is a coincidence of
this being a plate-bending study, not the same mechanism.

-----------------------------------------------------------------------------
WHY THERE IS NO CLOSED-FORM ANALYTICAL SOLUTION USED HERE
-----------------------------------------------------------------------------
Same reasoning as the sinusoidal/triangular FFFF scripts: the SS-SS-SS-SS
Navier series works because each sine-product mode individually satisfies
the simply-supported edge conditions, so any load's Fourier coefficients
q_mn (here, the patch-load coefficients from Timoshenko & Woinowsky-Krieger,
"Theory of Plates and Shells", Art. 32) can be superposed term-by-term. That
does not carry over to clamped edges regardless of load shape, and I do not
have the coupled built-in-edge series derivation memorized precisely enough
to transcribe with confidence. So the reference field here is instead a
direct numerical solve of D*grad^4(w)=q with w=0 and dw/dn=0 imposed at
every boundary node -- independently checkable via grid refinement, not
resting on a transcribed formula.

CAVEAT SPECIFIC TO THIS LOAD (found and fixed during validation, see chat
reply for the numbers): the patch load is a hard step function, and its
edges (at x = a/2 +/- PATCH_U/2, etc.) can fail a naive floating-point `>=`/
`<=` boundary comparison asymmetrically (e.g. 0.7 evaluating as
0.7000000000000001), silently breaking left-right symmetry of the solution.
Both `partial_patch_load_nd` (used in training) and
`partial_patch_load_nd_np` (used in the FD reference solve) now include a
small epsilon tolerance on the patch-edge comparisons to avoid this -- this
is a correctness fix relative to the original vanilla/FDM-PINN scripts'
patch-load function, not a change in the load's physical definition.
Also note: because the load is discontinuous, the FD reference solve
converges more slowly under grid refinement than it does for the smooth
sinusoidal/triangular loads (still visibly moving at 101x101 in my checks)
-- if you need a tight digit, refine nx/ny well beyond the 51x51 used here
for training collocation.

-----------------------------------------------------------------------------
WHAT CHANGED VS. THE SS-SS-SS-SS (vanilla FDM-PINN) PARTIAL-PATCH SCRIPT
-----------------------------------------------------------------------------
  - BC_CASE: "FFFF" (all four edges clamped/fixed/built-in), vs "SSSS".
  - Envelope: (xi*(1-xi))**2 * (eta*(1-eta))**2, instead of
    xi*(1-xi)*eta*(1-eta) -- double root at each edge hard-enforces both
    w=0 and dw/dn=0 simultaneously. Still evaluated fine on ghost points
    outside [0,a]x[0,b] -- still just a smooth polynomial continuation.
  - `fd_derivatives` now also returns w_x, w_y (first-derivative fields),
    needed for the new slope_bc_loss diagnostic below; everything else in
    that function is untouched.
  - Loss: total_loss = pde_loss only (no soft moment_loss -- no natural BC
    remains once both essential clamped conditions are hard-enforced).
    bc_loss (w residual) and the new slope_bc_loss (dw/dn residual) are
    kept as pure diagnostics, should be ~0.
  - Analytical Navier series (analytical_w_nd etc., _M/_N/_q_mn/_W_mn
    machinery) replaced by `solve_fd_reference_clamped`, called once with
    `partial_patch_load_nd_np` as the load function.
  - MODEL_NAME -> "FDM_PINN_FFFF_Clamped_HardBC_partial_patch" (own
    results/ folder).

Outputs (all written to ./results/<MODEL_NAME>/):
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python fdm_pinn_partial_patch_FFFF_clamped.py
"""

import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "FDM_PINN_FFFF_Clamped_HardBC_partial_patch"  # >>> VARIANT POINT
LOADING_CASE = "partial_patch"       # centered rectangular patch load
BC_CASE = "FFFF"                     # all four edges clamped / fixed / built-in

# Patch size as a fraction of the plate dimensions (unchanged: 0.4 x 0.4)
PATCH_U_FRAC = 0.4   # patch width along x, as a fraction of a
PATCH_V_FRAC = 0.4   # patch height along y, as a fraction of b

PATCH_EPS = 1e-9     # floating-point tolerance on the patch-edge comparisons
                      # (fixes a symmetry-breaking bug found during validation)

OUT_DIR = os.path.join("results", MODEL_NAME)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 45000
PATIENCE = 6000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

PATCH_U = PATCH_U_FRAC * a_nd   # patch width along x (non-dim)
PATCH_V = PATCH_V_FRAC * b_nd   # patch height along y (non-dim)

# ============================================================
# DOMAIN
# ============================================================
nx, ny = 51, 51
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

dx = a_nd / (nx - 1)
dy = b_nd / (ny - 1)
h_nd = dx   # uniform grid (nx == ny, a_nd == b_nd here) -- used by the FD reference solve

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept for the diagnostic bc_loss / slope_bc_loss -----
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)

bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)

# --- per-edge indices, reused here for the SLOPE diagnostic ----------------
# (for a clamped plate there is no natural/moment BC left to soft-penalize;
# these index sets are repurposed to report dw/dn residuals per edge pair)
edge_x0 = np.where(grid_points_nd[:, 0] == 0)[0]
edge_xa = np.where(grid_points_nd[:, 0] == a_nd)[0]
edge_y0 = np.where(grid_points_nd[:, 1] == 0)[0]
edge_yb = np.where(grid_points_nd[:, 1] == b_nd)[0]

edge_x_indices = np.unique(np.concatenate([edge_x0, edge_xa]))  # normal = x
edge_y_indices = np.unique(np.concatenate([edge_y0, edge_yb]))  # normal = y

edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)

# ============================================================
# GHOST-EXTENDED GRID (for the FDM-PINN derivative machinery -- UNCHANGED
# from the vanilla FDM-PINN partial-patch script)
# ============================================================
GHOST = 3

x_ext_1d = (np.arange(-GHOST, nx + GHOST)) * dx
y_ext_1d = (np.arange(-GHOST, ny + GHOST)) * dy
X_ext, Y_ext = np.meshgrid(x_ext_1d, y_ext_1d)

X_ext_flat = torch.tensor(X_ext.reshape(-1, 1), dtype=DTYPE, device=DEVICE)
Y_ext_flat = torch.tensor(Y_ext.reshape(-1, 1), dtype=DTYPE, device=DEVICE)
EXT_NY = ny + 2 * GHOST
EXT_NX = nx + 2 * GHOST

X_nd_t = torch.tensor(X_nd, dtype=DTYPE, device=DEVICE)
Y_nd_t = torch.tensor(Y_nd, dtype=DTYPE, device=DEVICE)

# --- 4th-order-accurate central finite-difference coefficients (Fornberg) --
C1 = torch.tensor([1 / 12, -8 / 12, 0.0, 8 / 12, -1 / 12], dtype=DTYPE, device=DEVICE)
C2 = torch.tensor([-1 / 12, 16 / 12, -30 / 12, 16 / 12, -1 / 12], dtype=DTYPE, device=DEVICE)
C4 = torch.tensor([-1 / 6, 2.0, -13 / 2, 28 / 3, -13 / 2, 2.0, -1 / 6], dtype=DTYPE, device=DEVICE)


def _apply_stencil_axis(arr, coeffs, axis, ghost, out_len):
    half = (coeffs.shape[0] - 1) // 2
    assert ghost >= half, "not enough ghost padding for this stencil"
    out = 0.0
    for k in range(-half, half + 1):
        c = coeffs[k + half]
        start = ghost + k
        if axis == 0:
            sl = arr[start:start + out_len, ...]
        else:
            sl = arr[:, start:start + out_len]
        out = out + c * sl
    return out


def d1_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C1, axis, ghost, out_len) / h


def d2_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C2, axis, ghost, out_len) / (h ** 2)


def d4_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C4, axis, ghost, out_len) / (h ** 4)


def fd_derivatives(model):
    """Single forward pass on the ghost-extended grid, then every
    derivative needed for training and post-processing as finite
    differences of that one pass. Returns (ny, nx)-shaped tensors:
    w, w_x, w_y, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy.
    (w_x, w_y are new relative to the vanilla FDM-PINN script -- needed
    here for the clamped-plate slope_bc_loss diagnostic.)"""
    w_ext = model(X_ext_flat, Y_ext_flat).reshape(EXT_NY, EXT_NX)

    w_rows = w_ext[GHOST:GHOST + ny, :]                              # (ny, nx+2G)
    w_x = d1_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)      # (ny, nx)
    w_xxxx = d4_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)
    w_xx = d2_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)

    w_cols = w_ext[:, GHOST:GHOST + nx]                               # (ny+2G, nx)
    w_y = d1_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)      # (ny, nx)
    w_yyyy = d4_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)
    w_yy = d2_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)

    w_yy_full = d2_axis(w_ext, axis=0, ghost=GHOST, out_len=ny, h=dy)
    w_xxyy = d2_axis(w_yy_full, axis=1, ghost=GHOST, out_len=nx, h=dx)

    w_y_full = d1_axis(w_ext, axis=0, ghost=GHOST, out_len=ny, h=dy)
    w_xy = d1_axis(w_y_full, axis=1, ghost=GHOST, out_len=nx, h=dx)

    w_phys = w_ext[GHOST:GHOST + ny, GHOST:GHOST + nx]

    return w_phys, w_x, w_y, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy


# ============================================================
# LOAD (centered rectangular patch, intensity q0)
# ============================================================
def partial_patch_load_nd(x, y):
    """Torch version, used inside the FDM-PINN loss. Includes a small
    epsilon tolerance on the patch-edge comparisons to avoid a
    floating-point symmetry-breaking bug (see module docstring)."""
    half_u = PATCH_U / 2.0
    half_v = PATCH_V / 2.0
    in_x = (x >= (a_nd / 2 - half_u - PATCH_EPS)) & (x <= (a_nd / 2 + half_u + PATCH_EPS))
    in_y = (y >= (b_nd / 2 - half_v - PATCH_EPS)) & (y <= (b_nd / 2 + half_v + PATCH_EPS))
    mask = (in_x & in_y).to(DTYPE)
    return q0_nd * mask


def partial_patch_load_nd_np(x, y):
    """NumPy version, used inside the finite-difference reference solve.
    Same epsilon-tolerance fix as the torch version above."""
    half_u = PATCH_U / 2.0
    half_v = PATCH_V / 2.0
    in_x = (x >= (a_nd / 2 - half_u - PATCH_EPS)) & (x <= (a_nd / 2 + half_u + PATCH_EPS))
    in_y = (y >= (b_nd / 2 - half_v - PATCH_EPS)) & (y <= (b_nd / 2 + half_v + PATCH_EPS))
    return q0_nd * (in_x & in_y).astype(np.float64)


Q_grid = partial_patch_load_nd(X_nd_t, Y_nd_t)   # (ny, nx), precomputed once


# ============================================================
# FINITE-DIFFERENCE REFERENCE SOLUTION (clamped plate, biharmonic PDE)
# -- a direct PDE solve, NOT the FDM-PINN training machinery above; see
# the "two different uses of finite differences" note in the module
# docstring.
# ============================================================
def solve_fd_reference_clamped(nx, ny, h, load_fn, D_nd=1.0):
    """
    Direct numerical ground truth for D*grad^4(w) = q(x,y) on [0,1]x[0,1]
    with w=0 and dw/dn=0 hard-enforced on all four edges (clamped/FFFF),
    via a standard central-difference biharmonic discretization. Identical
    method to the sinusoidal/triangular FFFF scripts, called here with
    `partial_patch_load_nd_np` as `load_fn`.

    This is NOT a closed-form / textbook formula -- it's a dense linear
    solve of the actual governing PDE and BCs. Because the patch load is
    discontinuous, convergence under grid refinement is slower here than
    for the smooth sinusoidal/triangular loads (validated separately;
    still visibly moving at 101x101 -- refine nx, ny well beyond 51x51 if
    you need a tight digit).

    Method: boundary nodes get w=0 directly; interior nodes use the
    standard 13-point biharmonic stencil, with second-orthogonal
    neighbors that land one step outside the domain replaced via the
    clamped-edge mirror/reflection identity w_{-1} = w_{+1} about the
    boundary node (the centered-difference statement of dw/dn = 0 there).
    """
    N = nx * ny

    def idx(j, i):
        return j * nx + i

    def reflect(k, n):
        if k == -1:
            return 1
        if k == n:
            return n - 2
        return k

    A = np.zeros((N, N), dtype=np.float64)
    b = np.zeros(N, dtype=np.float64)

    for j in range(ny):
        for i in range(nx):
            row = idx(j, i)
            if i == 0 or i == nx - 1 or j == 0 or j == ny - 1:
                A[row, row] = 1.0
                b[row] = 0.0
                continue

            x = i * h
            y = j * h
            b[row] = load_fn(np.array([x]), np.array([y]))[0] / D_nd * h ** 4

            def add(jj, ii, coeff):
                jj_r = reflect(jj, ny)
                ii_r = reflect(ii, nx)
                A[row, idx(jj_r, ii_r)] += coeff

            add(j, i, 20.0)
            add(j, i + 1, -8.0); add(j, i - 1, -8.0)
            add(j + 1, i, -8.0); add(j - 1, i, -8.0)
            add(j + 1, i + 1, 2.0); add(j + 1, i - 1, 2.0)
            add(j - 1, i + 1, 2.0); add(j - 1, i - 1, 2.0)
            add(j, i + 2, 1.0); add(j, i - 2, 1.0)
            add(j + 2, i, 1.0); add(j - 2, i, 1.0)

    w_vec = np.linalg.solve(A, b)
    return w_vec.reshape(ny, nx)


print("Solving finite-difference clamped-plate reference field "
      f"({nx}x{ny} dense linear system, {LOADING_CASE} load) -- "
      "this is a one-time setup cost...")
_fd_t0 = time.perf_counter()
w_ref_nd = solve_fd_reference_clamped(nx, ny, h_nd, partial_patch_load_nd_np, D_nd)
print(f"FD reference solve done in {time.perf_counter() - _fd_t0:.2f}s")

# Derivative fields for moments, via numpy central differences on the FD
# solution (np.gradient uses one-sided differences at the domain edges;
# small standard numerical error at the boundary only, doesn't affect the
# interior/center comparisons this script reports).
w_ref_y, w_ref_x = np.gradient(w_ref_nd, h_nd, h_nd)
w_ref_yy, w_ref_xy_from_y = np.gradient(w_ref_y, h_nd, h_nd)
w_ref_yx_from_x, w_ref_xx = np.gradient(w_ref_x, h_nd, h_nd)
w_ref_xy_nd = 0.5 * (w_ref_xy_from_y + w_ref_yx_from_x)

Mx_ref_nd = -(w_ref_xx + nu_nd * w_ref_yy)
My_ref_nd = -(w_ref_yy + nu_nd * w_ref_xx)
Mxy_ref_nd = -(1 - nu_nd) * w_ref_xy_nd

# ============================================================
# MODEL   (SiLU MLP; raw (xi, eta) input; SQUARED envelope hard-enforces
#          w=0 AND dw/dn=0 on all four edges -- the clamped/FFFF condition)
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3):
        super().__init__()
        layers = []
        in_dim = 2
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def forward(self, x, y):
        h = torch.cat([x / a_nd, y / b_nd], dim=1)
        h = self.hidden(h)
        raw = self.out(h)

        # Hard-constrained CLAMPED BCs: (xi(1-xi))^2 has a double root at
        # xi=0 and xi=1, so both the envelope and its derivative vanish
        # there (and likewise in eta) -- w=0 AND dw/dn=0 on every edge,
        # exactly, regardless of what raw is. Still fine on ghost points
        # outside [0,a]x[0,b] -- still just a smooth polynomial
        # continuation, no discontinuity, no special-casing needed.
        xi = x / a_nd
        eta = y / b_nd
        envelope = (xi * (1 - xi)) ** 2 * (eta * (1 - eta)) ** 2
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS (4th-order derivatives via finite differences, not autograd)
# ============================================================
def compute_loss():
    w_grid, w_x, w_y, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy = fd_derivatives(model)

    biharmonic = w_xxxx + 2 * w_xxyy + w_yyyy
    residual = (biharmonic - Q_grid).reshape(-1, 1)
    pde_loss = torch.mean(residual[int_idx] ** 2)

    w_flat = w_grid.reshape(-1, 1)
    # DIAGNOSTICS ONLY -- both should be ~0 exactly, by construction of the
    # squared envelope. Neither is in `total`.
    bc_loss = torch.mean(w_flat[bc_idx] ** 2)                                    # w=0 residual

    w_x_flat = w_x.reshape(-1, 1)
    w_y_flat = w_y.reshape(-1, 1)
    slope_bc_loss = (torch.mean(w_x_flat[edge_x_idx] ** 2)
                      + torch.mean(w_y_flat[edge_y_idx] ** 2))                   # dw/dn=0 residual

    # No natural (moment) BC remains for a fully clamped plate -- both
    # essential conditions are already hard-enforced by the envelope.
    total = pde_loss
    return total, pde_loss, bc_loss, slope_bc_loss, w_flat


# ============================================================
# TRAINING LOOP with computational-cost logging + early stopping
# ============================================================
epoch_history, loss_history, pde_history, bc_history, slope_bc_history = [], [], [], [], []
w_center_history, epoch_time_history = [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc, slope_bc, w_pred = compute_loss()
    total.backward()
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    slope_bc_history.append(slope_bc.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss {loss_val:.6e} (pde {pde.item():.3e}) "
              f"hardBC_w_residual {bc.item():.3e} hardBC_slope_residual {slope_bc.item():.3e} "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    if loss_val < best_loss:
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            converged_epoch = epoch
            break

total_train_time = time.perf_counter() - train_start
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_bc_w_residual_diagnostic": bc_history,
    "hard_bc_slope_residual_diagnostic": slope_bc_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS BALANCE DIAGNOSTIC
# ============================================================
final_pde = pde_history[-1]
final_bc, final_slope_bc = bc_history[-1], slope_bc_history[-1]
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "final_pde_loss": final_pde,
    "final_hard_bc_w_residual_diagnostic": final_bc,
    "final_hard_bc_slope_residual_diagnostic": final_slope_bc,
    "note": "total_loss == pde_loss for FFFF: both essential BCs are hard-enforced, no soft moment term.",
}
if final_bc > 1e-20 or final_slope_bc > 1e-20:
    print(f"WARNING: hard-BC residuals not ~0 (w: {final_bc:.3e}, slope: {final_slope_bc:.3e}) -- "
          f"double-check the envelope function / grid point exactness (float64 expected).")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "device": str(DEVICE),
    "epochs_run": len(epoch_history),
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,  # deprecated alias, same value
    "mean_time_per_epoch_s": float(np.mean(epoch_time_history)),
    "median_time_per_epoch_s": float(np.median(epoch_time_history)),
    "total_training_time_s": total_train_time,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "ghost_points_per_side": GHOST,
    "forward_pass_points": EXT_NX * EXT_NY,
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My, Mxy) via the same
# FDM-PINN derivative machinery used in training.
# ============================================================
def compute_full_fields():
    with torch.no_grad():
        w_grid, _, _, w_xx, w_yy, _, _, _, w_xy = fd_derivatives(model)
        Mx = -(w_xx + nu_nd * w_yy)
        My = -(w_yy + nu_nd * w_xx)
        Mxy = -(1 - nu_nd) * w_xy

    w_grid_np = w_grid.cpu().numpy()
    Mx_np = Mx.cpu().numpy()
    My_np = My.cpu().numpy()
    Mxy_np = Mxy.cpu().numpy()
    return w_grid_np, Mx_np, My_np, Mxy_np


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd = compute_full_fields()

w_dim_pred, w_dim_ref = w_pred_nd * W, w_ref_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_ref = Mx_pred_nd * moment_scale, Mx_ref_nd * moment_scale
My_dim_pred, My_dim_ref = My_pred_nd * moment_scale, My_ref_nd * moment_scale
Mxy_dim_pred, Mxy_dim_ref = Mxy_pred_nd * moment_scale, Mxy_ref_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_ref
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_ref)
My_error = np.abs(My_dim_pred - My_dim_ref)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_ref)

# ============================================================
# ERROR METRICS (vs. the FD reference field, NOT a closed-form solution)
# ============================================================
ix, iy = nx // 2, ny // 2
w_pred_center, w_ref_center = w_dim_pred[iy, ix], w_dim_ref[iy, ix]
center_abs_err = abs(w_pred_center - w_ref_center)
center_pct_err = center_abs_err / abs(w_ref_center) * 100

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_ref) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_ref) / np.linalg.norm(w_dim_ref)

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "reference_type": "finite_difference_biharmonic_solve",
    "reference_grid": f"{nx}x{ny}",
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_ref_m": float(w_ref_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": float(np.max(np.abs(w_dim_pred[0, :]))
                                         + np.max(np.abs(w_dim_pred[-1, :]))
                                         + np.max(np.abs(w_dim_pred[:, 0]))
                                         + np.max(np.abs(w_dim_pred[:, -1]))),
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_ref": w_dim_ref.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_ref": Mx_dim_ref.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_ref": My_dim_ref.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_ref": Mxy_dim_ref.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_ref"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_ref"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
plt.axhline(y=w_ref_center, linestyle="--", linewidth=2, color="k", label="FD Reference")
plt.xlabel("Epoch"); plt.ylabel("Center Deflection w (m)")
plt.title("Convergence of Center Deflection")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_ref_center), linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total Loss (= PDE Loss)")
plt.plot(df_hist["epoch"], df_hist["hard_bc_w_residual_diagnostic"],
         label="Hard-BC w residual (diagnostic, ~0)", alpha=0.6, linestyle=":")
plt.plot(df_hist["epoch"], df_hist["hard_bc_slope_residual_diagnostic"],
         label="Hard-BC slope residual (diagnostic, ~0)", alpha=0.6, linestyle=":")
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curve")
plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_ref, "FD Reference Plate Deflection", "Deflection (m)",
             "Reference_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_ref| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_ref", "w_pred - w_ref (m)",
             "Absolute_Error_w_pred_-_w_ref.png", cmap="coolwarm")

save_contour(Mx_dim_ref, "FD Reference Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x_Reference.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_ref| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_ref, "FD Reference Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y_Reference.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_ref| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_ref, "FD Reference Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy_Reference.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_ref| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_ref.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_ref.min(), w_dim_pred.min()), max(w_dim_ref.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("FD reference displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("FD Reference vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Reference_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}, {BC_CASE}")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_ref, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_ref, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_ref, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_ref, Mxy_dim_pred, Mxy_error),
]
for r, (label, ref, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"FD Reference {label}", ref, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading (u={PATCH_U_FRAC}a, v={PATCH_V_FRAC}b), "
             f"{BC_CASE} (summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Total training time: {total_train_time:.2f}s | "
      f"Epochs to early-stop: {converged_epoch} | "
      f"Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Total training time: {total_train_time:.2f}s | Epochs: {converged_epoch}")

Solving finite-difference clamped-plate reference field (51x51 dense linear system, partial_patch load) -- this is a one-time setup cost...
FD reference solve done in 0.44s
Epoch 0 Loss 1.061868e-01 (pde 1.062e-01) hardBC_w_residual 0.000e+00 hardBC_slope_residual 2.826e-19 Center 2.768521e-04 (330.99 ms/epoch)
Epoch 100 Loss 7.853466e-02 (pde 7.853e-02) hardBC_w_residual 0.000e+00 hardBC_slope_residual 2.032e-17 Center 5.174451e-04 (10.49 ms/epoch)
Epoch 200 Loss 7.823505e-02 (pde 7.824e-02) hardBC_w_residual 0.000e+00 hardBC_slope_residual 1.838e-17 Center 5.181836e-04 (11.50 ms/epoch)
Epoch 300 Loss 7.588291e-02 (pde 7.588e-02) hardBC_w_residual 0.000e+00 hardBC_slope_residual 5.037e-18 Center 5.198345e-04 (12.62 ms/epoch)
Epoch 400 Loss 5.832209e-02 (pde 5.832e-02) hardBC_w_residual 0.000e+00 hardBC_slope_residual 9.681e-16 Center 5.502330e-04 (11.93 ms/epoch)
Epoch 500 Loss 5.783338e-02 (pde 5.783e-02) hardBC_w_residual 0.000e+00 hardBC_slope_residual 9.216e-16 Center 5.485133e-04

C:\Users\vivek\AppData\Local\Temp\ipykernel_15624\3673306687.py:539: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_locatio

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'FDM_PINN_FFFF_Clamped_HardBC_partial_patch', 'loading_case': 'partial_patch', 'bc_case': 'FFFF', 'final_pde_loss': 0.0028397197801970607, 'final_hard_bc_w_residual_diagnostic': 0.0, 'final_hard_bc_slope_residual_diagnostic': 2.296020698382383e-14, 'note': 'total_loss == pde_loss for FFFF: both essential BCs are hard-enforced, no soft moment term.'}
Saved cost_metrics.json: {'model': 'FDM_PINN_FFFF_Clamped_HardBC_partial_patch', 'loading_case': 'partial_patch', 'bc_case': 'FFFF', 'device': 'cuda', 'epochs_run': 45000, 'epochs_to_early_stop': 45000, 'epochs_to_convergence': 45000, 'mean_time_per_epoch_s': 0.011315682197385468, 'median_time_per_epoch_s': 0.011181049922015518, 'total_training_time_s': 559.0728716999292, 'peak_gpu_memory_mb': 29.947392, 'n_trainable_params': 8578, 'ghost_points_per_side': 3, 'forward_pass_points': 3249}
Saved error_metrics.json: {'model': 'FDM_PINN_FFFF_Clamped_HardBC_partial_patch', 'loadin

In [1]:
"""
FDM-PINN (raw (x,y) input, FINITE-DIFFERENCE spatial derivatives instead of
Automatic Differentiation) for Kirchhoff-Love plate bending under a
PARTIALLY-DISTRIBUTED (centered rectangular patch) load, all-FOUR-edges-
CLAMPED / FIXED (FFFF).

>>> THIS FILE = the FFFF + L-BFGS-polish script, with ONLY the spatial
>>> derivative computation replaced: every torch.autograd.grad call that
>>> built w_x, w_y, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy is GONE.
>>> Everything else -- loading, network architecture (PlatePINN class is
>>> byte-for-byte the same), hard-BC envelope, analytical (beam-function
>>> Ritz) solution, Adam+L-BFGS training with min-delta early stopping,
>>> cost/error diagnostics, and plotting -- is UNCHANGED.

>>> HOW THE FDM RESIDUAL WORKS:
>>> The original script called the network once at each of the 51x51
>>> collocation points with requires_grad_(True) on (x,y), then used nested
>>> torch.autograd.grad calls to differentiate the network's OWN OUTPUT
>>> symbolically up to 4th order. FDM-PINN instead evaluates the network's
>>> forward pass on a slightly larger, GHOST-PADDED grid (2 extra points on
>>> every side, same uniform spacing h as the original 51x51 grid -- so the
>>> ghost points sit at x = -2h, -h and x = a+h, a+2h, i.e. just outside
>>> [0,a], same idea in y) and estimates every derivative from NEIGHBORING
>>> grid-point VALUES using standard central-difference stencils:
>>>     w_x  ~ (w(x+h,y) - w(x-h,y)) / (2h)
>>>     w_xx ~ (w(x+h,y) - 2w(x,y) + w(x-h,y)) / h^2
>>>     w_xy ~ (w(x+h,y+h) - w(x+h,y-h) - w(x-h,y+h) + w(x-h,y-h)) / (4h^2)
>>>     w_xxxx + 2 w_xxyy + w_yyyy  ~  the standard 13-point biharmonic
>>>         stencil: [20 w_C - 8(w_N+w_S+w_E+w_W) + 2(w_NE+w_NW+w_SE+w_SW)
>>>                    + (w_N2+w_S2+w_E2+w_W2)] / h^4
>>> Evaluating the network is still fully differentiable w.r.t. its OWN
>>> WEIGHTS (that's all backprop -- i.e. total.backward() during training --
>>> ever needed), so nothing about the training loop changes; the only
>>> thing that changed is that the SPATIAL derivatives feeding into the PDE
>>> residual are now finite-difference ESTIMATES of the network's output
>>> instead of EXACT symbolic derivatives of it.
>>> The entire ghost-padded grid needs only ONE forward pass per training
>>> step (batched, same as before) -- the 13-point stencil then reads off
>>> neighbor VALUES via array slicing, no extra network evaluations.
>>>
>>> Stencil formulas and the ghost-grid index arithmetic used below were
>>> verified in the sandbox against SymPy-exact derivatives of a smooth
>>> test polynomial (w = x^4+y^4+x^2y^2) before being used here: the
>>> 13-point biharmonic stencil matched the exact value to ~1e-7 absolute
>>> error, and the 5-point w_xx/w_yy/w_xy/w_x/w_y stencils matched to
>>> O(h^2) ~ 8e-4 (the expected truncation error of a 2nd-order-accurate
>>> central-difference scheme at h=1/50 -- not a bug, just finite-h error
>>> that a spectral/autograd derivative wouldn't have).

Load definition (unchanged -- origin O at plate center, patch of width u
along x and v along y, centered on the plate, intensity q0 uniform inside
the patch and zero outside):
    q(x,y) = q0   for |x - a/2| <= u/2  AND  |y - b/2| <= v/2
    q(x,y) = 0    otherwise
Patch size used here: u = 0.4*a, v = 0.4*b (PATCH_U_FRAC / PATCH_V_FRAC).

-----------------------------------------------------------------------------
ANALYTICAL SOLUTION -- unchanged from the FFFF script (beam-function
Rayleigh-Ritz/Galerkin, clamped-clamped edges). See that script's docstring
for the full derivation; summary:
    w(x,y) = sum_{m,n=1}^{M} C_mn X_m(x) Y_n(y)
with X_m, Y_n the normalized clamped-clamped BEAM eigenfunctions (roots of
cos(lam)cosh(lam)=1), and C_mn solved from the M^2 x M^2 Rayleigh-Ritz
linear system D[C_pq b_p^4 ab + 2 sum_mn C_mn G_mp G_nq + C_pq a g_q^4 b] = Q_pq.
Verified two ways: (1) uniform-load limit reproduces the tabulated
clamped-square-plate benchmark (Timoshenko & Woinowsky-Krieger) to ~1%,
(2) the actual 0.4a x 0.4b patch-load case cross-checked against an
independent finite-difference biharmonic solve to <0.1%.
-----------------------------------------------------------------------------
ALL FIXES CARRIED OVER:
  2. PDE residual no longer divided by an arbitrary 100.
  4. SiLU activation.
  6. Mxy analytical formula's 1/(a*b)-type factor confirmed present.
  7-8. Field reshape/indexing uses (ny, nx) consistently (row=y, col=x) --
       this convention is now shared by the FD ghost grid too (see below).
  9. epochs_to_convergence renamed to epochs_to_early_stop (old key kept).
 10. loss_balance.json diagnostic kept (trivially 100% pde_loss for FFFF).
 11. HARD BC: w=0 AND dw/dn=0 hard-enforced via the squared envelope
     [(x/a)(1-x/a)]^2 * [(y/b)(1-y/b)]^2 -- unchanged; bc_loss and
     slope_loss are diagnostics only, now computed from the FD-estimated
     w and w_x/w_y fields instead of autograd ones (still ~0 by
     construction, since the envelope forces it structurally, independent
     of how derivatives are estimated).
  B. Min-delta early stopping: a new "best" requires >= MIN_DELTA_REL
     relative improvement.
  C. L-BFGS polish phase after Adam, same loss, strong-Wolfe line search.

NOTE ON THE PDE FORCING TERM: unchanged -- the patch load is a hard
rectangular indicator function, evaluated forward-only (never
differentiated) at the interior collocation points.
-----------------------------------------------------------------------------

Outputs (all written to ./results/<MODEL_NAME>_<LOADING_CASE>_<BC_TYPE>/):
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python fdm_pinn_partial_patch_hardBC_FFFF.py
"""

import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.optimize import brentq
from scipy.integrate import quad

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "FDM_PINN"              # >>> CHANGED (was "Vanilla_PINN_HardBC") -- autograd derivatives replaced by FD stencils
LOADING_CASE = "partial_patch"       # centered rectangular patch load
BC_TYPE = "FFFF"                     # all four edges fixed/clamped
N_MODES = 24                         # beam-function modes per direction (analytical solution)

# Patch size as a fraction of the plate dimensions (confirmed with user: 0.4 x 0.4)
PATCH_U_FRAC = 0.4   # patch width along x, as a fraction of a
PATCH_V_FRAC = 0.4   # patch height along y, as a fraction of b

# NOTE: OUT_DIR includes LOADING_CASE and BC_TYPE so results don't overwrite
# the autograd-based FFFF results from the other script.
OUT_DIR = os.path.join("results", f"{MODEL_NAME}_{LOADING_CASE}_{BC_TYPE}")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 15000
PATIENCE = 3000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

# Min-delta early stopping -- a new "best" requires at least this RELATIVE
# improvement over the previous best, so noisy micro-improvements don't
# reset the patience counter and mask a genuine plateau.
MIN_DELTA_REL = 1e-4

# L-BFGS polish phase (after Adam)
LBFGS_MAX_ITER = 500          # internal L-BFGS iterations per outer step
LBFGS_OUTER_STEPS = 4         # how many times to call optimizer.step(closure)
LBFGS_HISTORY_SIZE = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

PATCH_U = PATCH_U_FRAC * a_nd   # patch width along x (non-dim)
PATCH_V = PATCH_V_FRAC * b_nd   # patch height along y (non-dim)

# ============================================================
# DOMAIN
# ============================================================
nx, ny = 51, 51
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

X_grid = torch.tensor(grid_points_nd, dtype=DTYPE, device=DEVICE)

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept only for the diagnostic bc_loss now ------------
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)

bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)  # not used by the FD residual (see below), kept for compatibility with the autograd-version index bookkeeping

# --- per-edge indices, kept only for the diagnostic SLOPE residual below.
edge_x0 = np.where(grid_points_nd[:, 0] == 0)[0]
edge_xa = np.where(grid_points_nd[:, 0] == a_nd)[0]
edge_y0 = np.where(grid_points_nd[:, 1] == 0)[0]
edge_yb = np.where(grid_points_nd[:, 1] == b_nd)[0]

edge_x_indices = np.unique(np.concatenate([edge_x0, edge_xa]))  # w_x = 0 here (slope diag.)
edge_y_indices = np.unique(np.concatenate([edge_y0, edge_yb]))  # w_y = 0 here (slope diag.)

edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)

# ============================================================
# >>> NEW: FINITE-DIFFERENCE GHOST-PADDED GRID (replaces the autograd
# derivative machinery). Same spacing h as the physical nx x ny grid, with
# 2 extra ghost points on every side so the 13-point biharmonic stencil is
# available at every INTERIOR physical point without any special-casing
# near the boundary. Verified (see module docstring) against SymPy-exact
# derivatives of a test polynomial before use.
# ============================================================
hx = a_nd / (nx - 1)
hy = b_nd / (ny - 1)
assert abs(hx - hy) < 1e-14, "FD stencils below assume a square uniform grid (hx == hy)"
h_fd = hx

nx_ext, ny_ext = nx + 4, ny + 4
x_ext_1d = np.linspace(-2 * hx, a_nd + 2 * hx, nx_ext)
y_ext_1d = np.linspace(-2 * hy, b_nd + 2 * hy, ny_ext)
X_ext, Y_ext = np.meshgrid(x_ext_1d, y_ext_1d)   # shape (ny_ext, nx_ext)
grid_ext_points = np.vstack([X_ext.ravel(), Y_ext.ravel()]).T
X_ext_grid = torch.tensor(grid_ext_points, dtype=DTYPE, device=DEVICE)  # no requires_grad needed -- FD, not autograd

# Physical grid coordinates for the strictly-interior region (used to
# evaluate the load q at the same points the PDE residual is computed at).
x_int_nd = x_nd[1:-1]   # nx-2 points
y_int_nd = y_nd[1:-1]   # ny-2 points
X_int_nd, Y_int_nd = np.meshgrid(x_int_nd, y_int_nd)     # shape (ny-2, nx-2)
grid_int_points = np.vstack([X_int_nd.ravel(), Y_int_nd.ravel()]).T
X_int_grid = torch.tensor(grid_int_points, dtype=DTYPE, device=DEVICE)


def _fd_fields(w_ext_2d):
    """Given the network output on the ghost-padded grid (shape
    (ny_ext, nx_ext)), return:
      w_phys_flat   : (nx*ny, 1) deflection on the ORIGINAL physical grid
                       (same flat ordering as X_grid / bc_idx / edge_*_idx)
      w_x_flat, w_y_flat : (nx*ny, 1) FD slope estimates on the physical grid
      biharm_interior : (ny-2, nx-2) FD estimate of w_xxxx+2w_xxyy+w_yyyy
                         at the strictly-interior physical points
      w_xx_flat, w_yy_flat, w_xy_flat : (nx*ny, 1) FD curvature/twist
                         estimates on the physical grid (for Mx, My, Mxy)
    """
    # --- physical-grid slice (1-ghost-layer central differences) --------
    center_full = w_ext_2d[2:ny + 2, 2:nx + 2]
    Ef = w_ext_2d[2:ny + 2, 3:nx + 3]
    Wf = w_ext_2d[2:ny + 2, 1:nx + 1]
    Nf = w_ext_2d[3:ny + 3, 2:nx + 2]
    Sf = w_ext_2d[1:ny + 1, 2:nx + 2]
    NEf = w_ext_2d[3:ny + 3, 3:nx + 3]
    NWf = w_ext_2d[3:ny + 3, 1:nx + 1]
    SEf = w_ext_2d[1:ny + 1, 3:nx + 3]
    SWf = w_ext_2d[1:ny + 1, 1:nx + 1]

    w_xx_full = (Ef - 2 * center_full + Wf) / hx ** 2
    w_yy_full = (Nf - 2 * center_full + Sf) / hy ** 2
    w_xy_full = (NEf - NWf - SEf + SWf) / (4 * hx * hy)
    w_x_full = (Ef - Wf) / (2 * hx)
    w_y_full = (Nf - Sf) / (2 * hy)

    w_phys_flat = center_full.reshape(-1, 1)
    w_x_flat = w_x_full.reshape(-1, 1)
    w_y_flat = w_y_full.reshape(-1, 1)
    w_xx_flat = w_xx_full.reshape(-1, 1)
    w_yy_flat = w_yy_full.reshape(-1, 1)
    w_xy_flat = w_xy_full.reshape(-1, 1)

    # --- strictly-interior 13-point biharmonic stencil -------------------
    center = w_ext_2d[3:ny + 1, 3:nx + 1]
    Ni = w_ext_2d[4:ny + 2, 3:nx + 1]
    Si = w_ext_2d[2:ny, 3:nx + 1]
    Ei = w_ext_2d[3:ny + 1, 4:nx + 2]
    Wi = w_ext_2d[3:ny + 1, 2:nx]
    NEi = w_ext_2d[4:ny + 2, 4:nx + 2]
    NWi = w_ext_2d[4:ny + 2, 2:nx]
    SEi = w_ext_2d[2:ny, 4:nx + 2]
    SWi = w_ext_2d[2:ny, 2:nx]
    N2i = w_ext_2d[5:ny + 3, 3:nx + 1]
    S2i = w_ext_2d[1:ny - 1, 3:nx + 1]
    E2i = w_ext_2d[3:ny + 1, 5:nx + 3]
    W2i = w_ext_2d[3:ny + 1, 1:nx - 1]

    biharm_interior = (20 * center - 8 * (Ni + Si + Ei + Wi)
                       + 2 * (NEi + NWi + SEi + SWi)
                       + (N2i + S2i + E2i + W2i)) / h_fd ** 4

    return w_phys_flat, w_x_flat, w_y_flat, biharm_interior, w_xx_flat, w_yy_flat, w_xy_flat


# ============================================================
# ANALYTICAL SOLUTION -- beam-function Rayleigh-Ritz/Galerkin, FFFF.
# UNCHANGED from the FFFF script; see module docstring for the derivation
# and the two independent verification checks.
# ============================================================

def _clamped_beam_eigenvalues(n_modes):
    """Positive roots lam_m of cos(lam)cosh(lam) = 1 (clamped-clamped beam)."""
    def f(lam):
        return np.cos(lam) * np.cosh(lam) - 1.0
    lams = []
    k = 1
    while len(lams) < n_modes:
        lo = (2 * k + 1) * np.pi / 2 - 1.5 if k > 1 else 4.0
        hi = (2 * k + 1) * np.pi / 2 + 1.5 if k > 1 else 5.6
        try:
            root = brentq(f, lo, hi)
            if len(lams) == 0 or root - lams[-1] > 1e-6:
                lams.append(root)
        except ValueError:
            pass
        k += 1
    return np.array(lams[:n_modes])


_lam = _clamped_beam_eigenvalues(N_MODES)
_beta = _lam / a_nd
_beta_y = _lam / b_nd
_sigma = (np.cosh(_lam) - np.cos(_lam)) / (np.sinh(_lam) - np.sin(_lam))


def _Xraw(m, x):
    bm = _beta[m]
    return np.cosh(bm * x) - np.cos(bm * x) - _sigma[m] * (np.sinh(bm * x) - np.sin(bm * x))


def _Xraw_p(m, x):
    bm = _beta[m]
    return bm * (np.sinh(bm * x) + np.sin(bm * x) - _sigma[m] * (np.cosh(bm * x) - np.cos(bm * x)))


def _Xraw_pp(m, x):
    bm = _beta[m]
    return bm ** 2 * (np.cosh(bm * x) + np.cos(bm * x) - _sigma[m] * (np.sinh(bm * x) + np.sin(bm * x)))


def _Yraw(n, y):
    bn = _beta_y[n]
    return np.cosh(bn * y) - np.cos(bn * y) - _sigma[n] * (np.sinh(bn * y) - np.sin(bn * y))


def _Yraw_p(n, y):
    bn = _beta_y[n]
    return bn * (np.sinh(bn * y) + np.sin(bn * y) - _sigma[n] * (np.cosh(bn * y) - np.cos(bn * y)))


def _Yraw_pp(n, y):
    bn = _beta_y[n]
    return bn ** 2 * (np.cosh(bn * y) + np.cos(bn * y) - _sigma[n] * (np.sinh(bn * y) + np.sin(bn * y)))


_norm_x = np.array([
    np.sqrt(quad(lambda x, m=m: _Xraw(m, x) ** 2, 0, a_nd, limit=200)[0] / a_nd)
    for m in range(N_MODES)
])
_norm_y = np.array([
    np.sqrt(quad(lambda y, n=n: _Yraw(n, y) ** 2, 0, b_nd, limit=200)[0] / b_nd)
    for n in range(N_MODES)
])


def _X(m, x):
    return _Xraw(m, x) / _norm_x[m]


def _Xp(m, x):
    return _Xraw_p(m, x) / _norm_x[m]


def _Xpp(m, x):
    return _Xraw_pp(m, x) / _norm_x[m]


def _Y(n, y):
    return _Yraw(n, y) / _norm_y[n]


def _Yp(n, y):
    return _Yraw_p(n, y) / _norm_y[n]


def _Ypp(n, y):
    return _Yraw_pp(n, y) / _norm_y[n]


_G = np.zeros((N_MODES, N_MODES))
for _m in range(N_MODES):
    for _p in range(N_MODES):
        _G[_m, _p] = quad(lambda x, m=_m, p=_p: _Xpp(m, x) * _X(p, x), 0, a_nd, limit=200)[0]

_H = np.zeros((N_MODES, N_MODES))
for _n in range(N_MODES):
    for _q in range(N_MODES):
        _H[_n, _q] = quad(lambda y, n=_n, q=_q: _Ypp(n, y) * _Y(q, y), 0, b_nd, limit=200)[0]

_beta4 = _beta ** 4
_gamma4 = _beta_y ** 4


def _idx(p, q):
    return p * N_MODES + q


def _assemble_and_solve_patch_load(u_frac, v_frac, q0):
    half_u, half_v = u_frac * a_nd / 2, v_frac * b_nd / 2
    x_lo, x_hi = a_nd / 2 - half_u, a_nd / 2 + half_u
    y_lo, y_hi = b_nd / 2 - half_v, b_nd / 2 + half_v

    Ip = np.array([quad(lambda x, p=p: _X(p, x), x_lo, x_hi, limit=200)[0] for p in range(N_MODES)])
    Iq = np.array([quad(lambda y, q=q: _Y(q, y), y_lo, y_hi, limit=200)[0] for q in range(N_MODES)])

    Npq = N_MODES * N_MODES
    K = np.zeros((Npq, Npq))
    Qvec = np.zeros(Npq)
    for p in range(N_MODES):
        for q in range(N_MODES):
            row = _idx(p, q)
            K[row, _idx(p, q)] += D_nd * (_beta4[p] * a_nd * b_nd + a_nd * _gamma4[q] * b_nd)
            K[row, :] += 2 * D_nd * np.einsum("m,n->mn", _G[:, p], _H[:, q]).ravel()
            Qvec[row] = q0 * Ip[p] * Iq[q]

    C = np.linalg.solve(K, Qvec).reshape(N_MODES, N_MODES)
    return C


_C_mn = _assemble_and_solve_patch_load(PATCH_U_FRAC, PATCH_V_FRAC, q0_nd)


def _series_field(x, y, kind="w"):
    out = np.zeros_like(x, dtype=float)
    for m in range(N_MODES):
        Xm = _X(m, x)
        Xmp = _Xp(m, x)
        Xmpp = _Xpp(m, x)
        for n in range(N_MODES):
            c = _C_mn[m, n]
            if c == 0.0:
                continue
            if kind == "w":
                out += c * Xm * _Y(n, y)
            elif kind == "Mx":
                out += -D_nd * c * (Xmpp * _Y(n, y) + nu_nd * Xm * _Ypp(n, y))
            elif kind == "My":
                out += -D_nd * c * (_Ypp(n, y) * Xm + nu_nd * Xmpp * _Y(n, y))
            elif kind == "Mxy":
                out += -D_nd * (1 - nu_nd) * c * Xmp * _Yp(n, y)
            else:
                raise ValueError(kind)
    return out


def analytical_w_nd(x, y):
    return _series_field(x, y, "w")


def analytical_Mx_nd(x, y):
    return _series_field(x, y, "Mx")


def analytical_My_nd(x, y):
    return _series_field(x, y, "My")


def analytical_Mxy_nd(x, y):
    return _series_field(x, y, "Mxy")


def partial_patch_load_nd(x, y):
    """Uniform load q0 over a centered rectangular patch of size
    PATCH_U x PATCH_V, zero elsewhere. x, y are torch tensors."""
    half_u = PATCH_U / 2.0
    half_v = PATCH_V / 2.0
    in_x = (x >= (a_nd / 2 - half_u)) & (x <= (a_nd / 2 + half_u))
    in_y = (y >= (b_nd / 2 - half_v)) & (y <= (b_nd / 2 + half_v))
    mask = (in_x & in_y).to(DTYPE)
    return q0_nd * mask


# ============================================================
# MODEL -- UNCHANGED (same architecture, same hard-BC envelope). Only the
# way its output is DIFFERENTIATED (in compute_loss / compute_full_fields
# below) has changed, not the network itself.
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3):
        super().__init__()
        layers = []
        in_dim = 2
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def forward(self, x, y):
        h = torch.cat([x / a_nd, y / b_nd], dim=1)
        h = self.hidden(h)
        raw = self.out(h)

        xi = x / a_nd
        eta = y / b_nd
        envelope = (xi * (1 - xi)) ** 2 * (eta * (1 - eta)) ** 2
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE (Adam phase)
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS -- >>> CHANGED: finite-difference spatial derivatives instead of
# autograd. ONE forward pass on the ghost-padded grid, then the 13-point
# biharmonic stencil (interior only) for the PDE residual, plus 5-point
# central differences (full physical grid) for the w/w_x/w_y diagnostics.
# ============================================================
def compute_loss():
    xg = X_ext_grid[:, 0:1]
    yg = X_ext_grid[:, 1:2]
    w_ext = model(xg, yg)
    w_ext_2d = w_ext.reshape(ny_ext, nx_ext)

    w_phys_flat, w_x_flat, w_y_flat, biharm_interior, _, _, _ = _fd_fields(w_ext_2d)

    xq = X_int_grid[:, 0:1]
    yq = X_int_grid[:, 1:2]
    q_interior = partial_patch_load_nd(xq, yq).reshape(ny - 2, nx - 2)

    pde = biharm_interior - q_interior
    pde_loss = torch.mean(pde ** 2)

    # DIAGNOSTIC ONLY -- both should be ~0 exactly, by construction of the
    # hard-BC envelope (independent of FD vs autograd). Neither is in
    # `total`; their gradients don't drive training.
    bc_loss = torch.mean(w_phys_flat[bc_idx] ** 2)
    slope_loss = (torch.mean(w_x_flat[edge_x_idx] ** 2)
                  + torch.mean(w_y_flat[edge_y_idx] ** 2))

    # No soft moment loss for FFFF -- unchanged from the FFFF script.
    total = pde_loss
    return total, pde_loss, bc_loss, slope_loss, w_phys_flat


# ============================================================
# ADAM TRAINING LOOP with computational-cost logging + min-delta early
# stopping
# ============================================================
epoch_history, loss_history, pde_history, bc_history, slope_history = [], [], [], [], []
w_center_history, epoch_time_history, phase_history = [], [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS
adam_epochs_run = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc, slope, w_pred = compute_loss()
    total.backward()
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    slope_history.append(slope.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)
    phase_history.append("adam")

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss {loss_val:.6e} (pde {pde.item():.3e}, "
              f"hardBC_w_residual {bc.item():.3e}, hardBC_slope_residual {slope.item():.3e}) "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    if loss_val < best_loss * (1.0 - MIN_DELTA_REL):
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping (Adam phase) triggered at epoch {epoch}")
            converged_epoch = epoch
            adam_epochs_run = epoch + 1
            break

adam_time = time.perf_counter() - train_start

if not os.path.exists(weights_path):
    torch.save(model.state_dict(), weights_path)

# ============================================================
# L-BFGS POLISH PHASE (unchanged mechanics -- operates on the same
# compute_loss(), which now internally uses FD instead of autograd)
# ============================================================
model.load_state_dict(torch.load(weights_path, map_location=DEVICE))

lbfgs_optimizer = torch.optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=LBFGS_MAX_ITER,
    history_size=LBFGS_HISTORY_SIZE,
    line_search_fn="strong_wolfe",
)

lbfgs_start = time.perf_counter()
lbfgs_best_loss = best_loss
lbfgs_iters_run = 0


def _lbfgs_closure():
    global lbfgs_best_loss, lbfgs_iters_run
    lbfgs_optimizer.zero_grad()
    total, pde, bc, slope, w_pred = compute_loss()
    total.backward()

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()
    epoch_history.append(adam_epochs_run + lbfgs_iters_run)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    slope_history.append(slope.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(float("nan"))
    phase_history.append("lbfgs")

    if loss_val < lbfgs_best_loss:
        lbfgs_best_loss = loss_val
        torch.save(model.state_dict(), weights_path)

    lbfgs_iters_run += 1
    return total


for outer_step in range(LBFGS_OUTER_STEPS):
    lbfgs_optimizer.step(_lbfgs_closure)
    print(f"L-BFGS outer step {outer_step + 1}/{LBFGS_OUTER_STEPS} done "
          f"(best loss so far: {lbfgs_best_loss:.6e})")

lbfgs_time = time.perf_counter() - lbfgs_start
total_train_time = adam_time + lbfgs_time
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "phase": phase_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_bc_w_residual_diagnostic": bc_history,
    "hard_bc_slope_residual_diagnostic": slope_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS BALANCE DIAGNOSTIC (trivially 100% pde_loss for FFFF)
# ============================================================
final_pde, final_bc, final_slope = pde_history[-1], bc_history[-1], slope_history[-1]
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "derivative_method": "finite_difference",
    "fd_step_h": h_fd,
    "final_pde_loss": final_pde,
    "pde_fraction_of_total": 1.0,
    "final_hard_bc_w_residual_diagnostic": final_bc,
    "final_hard_bc_slope_residual_diagnostic": final_slope,
}
if final_bc > 1e-20 or final_slope > 1e-20:
    print(f"WARNING: hard BC residual diagnostics are not ~0 "
          f"(w: {final_bc:.3e}, slope: {final_slope:.3e}) -- "
          f"double-check the envelope function / grid point exactness (float64 expected).")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
adam_epoch_times = [t for t, p in zip(epoch_time_history, phase_history) if p == "adam"]
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "derivative_method": "finite_difference",
    "device": str(DEVICE),
    "adam_epochs_run": adam_epochs_run,
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,
    "lbfgs_outer_steps": LBFGS_OUTER_STEPS,
    "lbfgs_max_iter_per_step": LBFGS_MAX_ITER,
    "lbfgs_closure_calls": lbfgs_iters_run,
    "mean_time_per_adam_epoch_s": float(np.mean(adam_epoch_times)) if adam_epoch_times else float("nan"),
    "median_time_per_adam_epoch_s": float(np.median(adam_epoch_times)) if adam_epoch_times else float("nan"),
    "adam_phase_time_s": adam_time,
    "lbfgs_phase_time_s": lbfgs_time,
    "total_training_time_s": total_train_time,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My, Mxy) -- >>> CHANGED:
# FD stencils (same _fd_fields helper used during training) instead of
# autograd.
# ============================================================
def compute_full_fields():
    with torch.no_grad():
        xg = X_ext_grid[:, 0:1]
        yg = X_ext_grid[:, 1:2]
        w_ext = model(xg, yg)
        w_ext_2d = w_ext.reshape(ny_ext, nx_ext)

        w_phys_flat, _, _, _, w_xx_flat, w_yy_flat, w_xy_flat = _fd_fields(w_ext_2d)

        Mx_flat = -(w_xx_flat + nu_nd * w_yy_flat)
        My_flat = -(w_yy_flat + nu_nd * w_xx_flat)
        Mxy_flat = -(1 - nu_nd) * w_xy_flat

        w_grid = w_phys_flat.cpu().numpy().reshape(ny, nx)
        Mx_grid = Mx_flat.cpu().numpy().reshape(ny, nx)
        My_grid = My_flat.cpu().numpy().reshape(ny, nx)
        Mxy_grid = Mxy_flat.cpu().numpy().reshape(ny, nx)
    return w_grid, Mx_grid, My_grid, Mxy_grid


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd = compute_full_fields()

w_anal_nd = analytical_w_nd(X_nd, Y_nd)
Mx_anal_nd = analytical_Mx_nd(X_nd, Y_nd)
My_anal_nd = analytical_My_nd(X_nd, Y_nd)
Mxy_anal_nd = analytical_Mxy_nd(X_nd, Y_nd)

w_dim_pred, w_dim_anal = w_pred_nd * W, w_anal_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_anal = Mx_pred_nd * moment_scale, Mx_anal_nd * moment_scale
My_dim_pred, My_dim_anal = My_pred_nd * moment_scale, My_anal_nd * moment_scale
Mxy_dim_pred, Mxy_dim_anal = Mxy_pred_nd * moment_scale, Mxy_anal_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_anal
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_anal)
My_error = np.abs(My_dim_pred - My_dim_anal)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_anal)

# ============================================================
# ERROR METRICS
# ============================================================
ix, iy = nx // 2, ny // 2
w_pred_center, w_anal_center = w_dim_pred[iy, ix], w_dim_anal[iy, ix]
center_abs_err = abs(w_pred_center - w_anal_center)
center_pct_err = center_abs_err / abs(w_anal_center) * 100

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_anal) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_anal_m": float(w_anal_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": float(np.max(np.abs(w_dim_pred[0, :]))
                                         + np.max(np.abs(w_dim_pred[-1, :]))
                                         + np.max(np.abs(w_dim_pred[:, 0]))
                                         + np.max(np.abs(w_dim_pred[:, -1]))),
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_anal": w_dim_anal.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_anal": Mx_dim_anal.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_anal": My_dim_anal.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_anal": Mxy_dim_anal.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_anal"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_anal"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING (unchanged)
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
lbfgs_start_epoch = adam_epochs_run
plt.axvline(x=lbfgs_start_epoch, color="gray", linestyle=":", linewidth=1, label="L-BFGS starts")
plt.axhline(y=w_anal_center, linestyle="--", linewidth=2, color="k", label="Analytical")
plt.xlabel("Epoch (Adam) / Closure call (L-BFGS)"); plt.ylabel("Center Deflection w (m)")
plt.title("Convergence of Center Deflection")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_anal_center), linewidth=2)
plt.axvline(x=lbfgs_start_epoch, color="gray", linestyle=":", linewidth=1, label="L-BFGS starts")
plt.yscale("log")
plt.xlabel("Epoch (Adam) / Closure call (L-BFGS)"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.legend(); plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total Loss (= PDE Loss)")
plt.plot(df_hist["epoch"], df_hist["hard_bc_w_residual_diagnostic"],
         label="Hard-BC w residual (diagnostic, ~0)", alpha=0.5, linestyle=":")
plt.plot(df_hist["epoch"], df_hist["hard_bc_slope_residual_diagnostic"],
         label="Hard-BC slope residual (diagnostic, ~0)", alpha=0.5, linestyle=":")
plt.axvline(x=lbfgs_start_epoch, color="gray", linestyle=":", linewidth=1, label="L-BFGS starts")
plt.yscale("log")
plt.xlabel("Epoch (Adam) / Closure call (L-BFGS)"); plt.ylabel("Loss"); plt.title("Training Loss Curve (FDM residual)")
plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_anal, "Analytical Plate Deflection", "Deflection (m)",
             "Analytical_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_anal| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_anal", "w_pred - w_anal (m)",
             "Absolute_Error_w_pred_-_w_anal.png", cmap="coolwarm")

save_contour(Mx_dim_anal, "Analytical Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x_Analytical.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_anal| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_anal, "Analytical Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y_Analytical.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_anal| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_anal, "Analytical Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy_Analytical.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_anal| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_anal.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_anal.min(), w_dim_pred.min()), max(w_dim_anal.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("Analytical displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("Analytical vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Analytical_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}, {BC_TYPE}, FDM residual")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_anal, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_anal, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_anal, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_anal, Mxy_dim_pred, Mxy_error),
]
for r, (label, anal, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"Analytical {label}", anal, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading (u={PATCH_U_FRAC}a, v={PATCH_V_FRAC}b), "
             f"{BC_TYPE}, FDM residual (summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Adam phase: {adam_time:.2f}s ({adam_epochs_run} epochs, early-stop epoch "
      f"{converged_epoch}) | L-BFGS phase: {lbfgs_time:.2f}s ({lbfgs_iters_run} closure calls) | "
      f"Total: {total_train_time:.2f}s | Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Adam phase: {adam_time:.2f}s ({adam_epochs_run} epochs) | "
      f"L-BFGS phase: {lbfgs_time:.2f}s ({lbfgs_iters_run} closure calls) | "
      f"Total: {total_train_time:.2f}s")

C:\Users\vivek\AppData\Local\Temp\ipykernel_14200\3470408314.py:366: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  np.sqrt(quad(lambda x, m=m: _Xraw(m, x) ** 2, 0, a_nd, limit=200)[0] / a_nd)
C:\Users\vivek\AppData\Local\Temp\ipykernel_14200\3470408314.py:366: IntegrationWarning: The maximum number of subdivisions (200) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  np.sqrt(quad(lambda x, m=m: _Xraw(m, x) ** 2, 0, a_nd, limit=200)[0] / a_nd)
C:\Users\vivek\AppData\Local\Temp\ipykernel_14200\3470408314.py:370: Integration

Epoch 0 Loss 9.645551e-02 (pde 9.646e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 9.502e-12) Center 2.768521e-04 (273.62 ms/epoch)
Epoch 100 Loss 7.541935e-02 (pde 7.542e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 1.558e-11) Center 4.786997e-04 (5.19 ms/epoch)
Epoch 200 Loss 7.493465e-02 (pde 7.493e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 1.531e-11) Center 4.791241e-04 (5.12 ms/epoch)
Epoch 300 Loss 5.886435e-02 (pde 5.886e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 1.751e-11) Center 4.961529e-04 (5.08 ms/epoch)
Epoch 400 Loss 5.336594e-02 (pde 5.337e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 1.976e-11) Center 5.094326e-04 (5.07 ms/epoch)
Epoch 500 Loss 5.284922e-02 (pde 5.285e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 1.984e-11) Center 5.105164e-04 (5.14 ms/epoch)
Epoch 600 Loss 5.200529e-02 (pde 5.201e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 1.998e-11) Center 5.126007e-04 (5.19 ms/epoch)
Epoch 

C:\Users\vivek\AppData\Local\Temp\ipykernel_14200\3470408314.py:634: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_locatio

L-BFGS outer step 1/4 done (best loss so far: 1.537423e-02)
L-BFGS outer step 2/4 done (best loss so far: 1.391014e-02)
L-BFGS outer step 3/4 done (best loss so far: 1.189900e-02)
L-BFGS outer step 4/4 done (best loss so far: 1.083475e-02)


C:\Users\vivek\AppData\Local\Temp\ipykernel_14200\3470408314.py:683: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_locatio

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'FDM_PINN', 'loading_case': 'partial_patch', 'bc_type': 'FFFF', 'derivative_method': 'finite_difference', 'fd_step_h': 0.02, 'final_pde_loss': 0.010834753405944732, 'pde_fraction_of_total': 1.0, 'final_hard_bc_w_residual_diagnostic': 0.0, 'final_hard_bc_slope_residual_diagnostic': 3.7239149114974433e-11}
Saved cost_metrics.json: {'model': 'FDM_PINN', 'loading_case': 'partial_patch', 'bc_type': 'FFFF', 'derivative_method': 'finite_difference', 'device': 'cuda', 'adam_epochs_run': 15000, 'epochs_to_early_stop': 15000, 'epochs_to_convergence': 15000, 'lbfgs_outer_steps': 4, 'lbfgs_max_iter_per_step': 500, 'lbfgs_closure_calls': 2115, 'mean_time_per_adam_epoch_s': 0.00506679488698331, 'median_time_per_adam_epoch_s': 0.005019500153139234, 'adam_phase_time_s': 86.14684140007012, 'lbfgs_phase_time_s': 34.81992679997347, 'total_training_time_s': 120.96676820004359, 'peak_gpu_memory_mb': 35.96288, 'n_trainable_params': 8578}
Save